### Phase 1 - Data Preparation
1. Load and Verify the Transactions

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("..") / "data"

print("Working directory:", Path.cwd())
print("File exists:", (DATA_DIR / "products.csv").exists())

Working directory: c:\School\420-975-VA_Unsupervised_ML\Final Project\boreal-reco-MatthewBraithwaite\notebooks
File exists: True


In [2]:
df_products = pd.read_csv(DATA_DIR / "products.csv")

display(df_products.head())
print(df_products.shape)
print(df_products.columns)

,Item,Item_EN,Category,Price
0,Steak de boeuf,Beef steak,Viandes,18.99
1,Saucisses,Sausages,Viandes,7.49
2,Poitrine de poulet,Chicken breast,Viandes,12.99
3,Bacon,Bacon,Viandes,6.99
4,Saumon frais,Fresh salmon,Poissons,16.49


(98, 4)
Index(['Item', 'Item_EN', 'Category', 'Price'], dtype='str')


In [3]:
# Load transaction data
df_transactions = pd.read_csv(
    DATA_DIR / "transactionsdata.zip",
    usecols=["TransactionID", "Date", "Item"]
)

display(df_transactions.head())
print(df_transactions.shape)

,TransactionID,Date,Item
0,100000,2026-06-19,Cidre de glace
1,100000,2026-06-19,Couches
2,100000,2026-06-19,Fromage cheddar
3,100000,2026-06-19,Lingettes pour bébé
4,100000,2026-06-19,Purée pour bébé


(8379837, 3)


In [4]:
# Verify the transaction dataset
print(f"Total rows: {df_transactions.shape[0]:,}")
print(f"Total baskets: {df_transactions['TransactionID'].nunique():,}")
print(f"Total products: {df_transactions['Item'].nunique():,}")

# Calculate average basket size
basket_sizes = df_transactions.groupby("TransactionID").size()

print(f"Average basket size: {basket_sizes.mean():.2f}")

Total rows: 8,379,837
Total baskets: 1,000,000
Total products: 98
Average basket size: 8.38


In [5]:
# Check for missing values and duplicate products within baskets
print(df_transactions[["TransactionID", "Item"]].isna().sum())

print(
    "Duplicate transaction-item pairs:",
    df_transactions.duplicated(subset=["TransactionID", "Item"]).sum()
)

TransactionID    0
Item             0
dtype: int64
Duplicate transaction-item pairs: 0


In [6]:
# Check memory usage of the transaction DataFrame
memory_mb = df_transactions.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Transaction DataFrame memory: {memory_mb:.2f} MB")

Transaction DataFrame memory: 374.27 MB


In [7]:
# Estimate the size of the Boolean basket matrix
total_cells = (
    df_transactions["TransactionID"].nunique()
    * df_transactions["Item"].nunique()
)

dense_memory_mb = total_cells / (1024 ** 2)

# Calculate how many positions will contain True
density = len(df_transactions) / total_cells * 100

print(f"Total Boolean cells: {total_cells:,}")
print(f"Estimated dense matrix memory: {dense_memory_mb:.2f} MiB")
print(f"Matrix density: {density:.2f}%")

Total Boolean cells: 98,000,000
Estimated dense matrix memory: 93.46 MiB
Matrix density: 8.55%


In [8]:
# Select 1,000 transaction IDs for my test
sample_ids = basket_sizes.index[:1000]

# Keep the transactions belonging to those baskets
df_sample = df_transactions[
    df_transactions["TransactionID"].isin(sample_ids)
]

# Familiar homework technique
baskets_sample = df_sample.groupby("TransactionID")["Item"].apply(list)

print(f"Sample baskets: {len(baskets_sample):,}")
print(f"Sample transaction rows: {len(df_sample):,}")

display(baskets_sample.head())

Sample baskets: 1,000
Sample transaction rows: 8,374


TransactionID
100000    [Cidre de glace, Couches, Fromage cheddar, Lin...
100001    [Biscuits, Bière, Bretzels, Croustilles, Froma...
100002    [Ail, Crevettes, Crème glacée, Noix mélangées,...
100003    [Algues nori, Beurre d'arachide, Bouillon à fo...
100004    [Café moulu, Croissants, Fromage suisse, Oeufs...
Name: Item, dtype: object

In [9]:
# Encode the sample baskets using my homework technique
from mlxtend.preprocessing import TransactionEncoder
import time

start_time = time.perf_counter()

te = TransactionEncoder()
encoded_sample = te.fit_transform(baskets_sample.tolist())

df_encoded_sample = pd.DataFrame(
    encoded_sample,
    columns=te.columns_,
    index=baskets_sample.index
)

elapsed_time = time.perf_counter() - start_time

# Verify dimensions and memory
print(f"Encoded matrix shape: {df_encoded_sample.shape}")
print(f"Encoding time: {elapsed_time:.4f} seconds")

sample_memory = df_encoded_sample.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Encoded matrix memory: {sample_memory:.4f} MiB")

display(df_encoded_sample.head())

Encoded matrix shape: (1000, 95)
Encoding time: 0.0032 seconds
Encoded matrix memory: 0.0982 MiB


,Ail,Algues nori,Avocat,Bacon,Bagels,Baguette,Bananes,Beurre,Beurre d'arachide,Biscuits,...,Sucre blanc,Thon en conserve,Thé vert,Tomates,Tortillas,Vin blanc,Vin rouge,Vinaigre balsamique vieilli,Wasabi,Yogourt grec
TransactionID,,,,,,,,,,,,,,,,,,,,,
100000,False,False,False,False,False,False,False,False,False,False,...,False,True,True,False,False,False,False,False,False,False
100001,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
100002,True,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
100003,False,True,False,False,False,False,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
100004,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [10]:
# Verify the encoded sample
print(f"Original product rows: {len(df_sample):,}")
print(f"Total True values: {df_encoded_sample.to_numpy().sum():,}")

# Identify products missing from the sample
all_products = set(df_transactions["Item"].unique())
sample_products = set(df_encoded_sample.columns)

missing_products = all_products.difference(sample_products)

print(f"Products missing from sample: {sorted(missing_products)}")

Original product rows: 8,374
Total True values: 8,374
Products missing from sample: ['Caviar', 'Huile de truffe', 'Safran']


In [11]:
# Explore integer positions for baskets and products
basket_codes, basket_ids = pd.factorize(df_sample["TransactionID"])
product_codes, product_names = pd.factorize(df_sample["Item"])

print("First five basket codes:", basket_codes[:5])
print("First five product codes:", product_codes[:5])

print("Unique baskets:", len(basket_ids))
print("Unique products:", len(product_names))

First five basket codes: [0 0 0 0 0]
First five product codes: [0 1 2 3 4]
Unique baskets: 1000
Unique products: 95


In [12]:
# Create an empty Boolean matrix for my sample
encoded_vectorized = np.zeros(
    (len(basket_ids), len(product_names)),
    dtype=bool
)

# Mark purchased products as True
encoded_vectorized[basket_codes, product_codes] = True

# Convert to my familiar Pandas DataFrame
df_encoded_vectorized = pd.DataFrame(
    encoded_vectorized,
    columns=product_names,
    index=basket_ids
)

# Compare with my original TransactionEncoder results
df_expected = df_encoded_sample.reindex(
    index=basket_ids,
    columns=product_names
)

print(f"Matrix shape: {df_encoded_vectorized.shape}")
print(f"Total True values: {df_encoded_vectorized.to_numpy().sum():,}")
print(f"Matches TransactionEncoder: {df_encoded_vectorized.equals(df_expected)}")

Matrix shape: (1000, 95)
Total True values: 8,374
Matches TransactionEncoder: True


In [13]:
# Start measuring
start_time = time.perf_counter()

# Use the full transaction dataset
basket_codes_full, basket_ids_full = pd.factorize(df_transactions["TransactionID"])
product_codes_full, product_names_full = pd.factorize(df_transactions["Item"])

# Create the full Boolean matrix
encoded_full = np.zeros(
    (len(basket_ids_full), len(product_names_full)),
    dtype=bool
)

# Set purchased combinations to True
encoded_full[basket_codes_full, product_codes_full] = True

# Create the DataFrame
df_encoded = pd.DataFrame(
    encoded_full,
    columns=product_names_full,
    index=basket_ids_full
)

# Stop measuring
elapsed_time = time.perf_counter() - start_time

In [14]:
# Verify the full Boolean matrix
print(f"Matrix shape: {df_encoded.shape}")
print(f"Total True values: {df_encoded.to_numpy().sum():,}")

# Measure execution time and actual DataFrame memory
print(f"Transformation time: {elapsed_time:.4f} seconds")

matrix_memory = df_encoded.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Boolean matrix memory: {matrix_memory:.2f} MiB")

# Verify that all products are represented
missing_products = set(df_products["Item"]).difference(df_encoded.columns)

print(f"Missing products: {sorted(missing_products)}")

Matrix shape: (1000000, 98)
Total True values: 8,379,837
Transformation time: 1.5874 seconds
Boolean matrix memory: 101.09 MiB
Missing products: []


In [15]:
# Calculate how often each product appears
product_counts = df_encoded.sum(axis=0).sort_values(ascending=False)

# Calculate product frequency as a percentage of baskets
df_frequencies = pd.DataFrame({
    "Basket Count": product_counts,
    "Frequency (%)": product_counts / len(df_encoded) * 100
})

# Display the 10 most frequent products
print("Top 10 Most Frequent Products")
display(df_frequencies.head(10).round(2))

# Display the 5 rarest products
print("5 Rarest Products")
display(
    df_frequencies.tail(5)
    .sort_values("Basket Count")
    .round(2)
)

Top 10 Most Frequent Products


,Basket Count,Frequency (%)
Sac réutilisable,900026,90.00
Biscuits,161439,16.14
Noix mélangées,155189,15.52
Beurre d'arachide,153726,15.37
Bière,150468,15.05
Savon à vaisselle,149820,14.98
Fromage cheddar,143457,14.35
Charbon de bois,140561,14.06
Papier hygiénique,139758,13.98
Céréales,139198,13.92


5 Rarest Products


,Basket Count,Frequency (%)
Foie gras,756,0.08
Vinaigre balsamique vieilli,779,0.08
Caviar,810,0.08
Huile de truffe,817,0.08
Safran,818,0.08


### Pre-Mining Prediction

- Prediction A - For Rare products such as Foie gras, Caviar, Truffle Oil, and Saffron, they will struggle to produce association rules because their individual support is below the lowest planned min_support=0.005 (0.5%). Apriori cannot generate frequent itemsets containing them at that threshold.

- Prediction B - Reusable Bag appears in 90% of baskets, I predict it will dominate the confidence rankings but may have lift values close to 1, indicating weak associations relative to its baseline popularity.

In [16]:
# Review the catalogue for duplicate product identities
display(
    df_products[
        ["Item", "Item_EN", "Category", "Price"]
    ].sort_values(["Category", "Item_EN"])
)

,Item,Item_EN,Category,Price
65,Bière,Beer,Boissons,14.99
69,Thé vert,Green tea,Boissons,5.99
68,Café moulu,Ground coffee,Boissons,11.99
73,Cidre de glace,Ice cider,Boissons,24.99
70,Jus d'orange,Orange juice,Boissons,4.99
...,...,...,...,...
19,Confiture de fraises,Strawberry jam,Épicerie,4.49
24,Riz à sushi,Sushi rice,Épicerie,6.99
13,Sauce tomate,Tomato sauce,Épicerie,3.79
27,Wasabi,Wasabi,Épicerie,4.49


In [17]:
# Check for repeated English product names
duplicate_names = df_products[
    df_products["Item_EN"].duplicated(keep=False)
]

display(duplicate_names)

,Item,Item_EN,Category,Price


### Duplicate Products

- I had considered `Baguette` and `Rustic Baguette`, however they might actually be different products; the rustic designation creates uncertainty. The price difference supports further investigation but does not prove they are different products. Merging them would change combined support, potentially different frequent itemsets, and different recommendations. Based on the available information, I decided not to merge these products because I cannot confirm they are identical, and merging them could distort my association rules.

### Phase 2 - Apriori mining

In [18]:
# Phase 2 - Apriori mining
from mlxtend.frequent_patterns import apriori, association_rules

# First experiment: 5% minimum support
start_time = time.perf_counter()

df_itemsets = apriori(
    df_encoded,
    min_support=0.05,
    use_colnames=True,
    low_memory=True
)

apriori_time = time.perf_counter() - start_time

print(f"Frequent itemsets: {len(df_itemsets):,}")
print(f"Apriori execution time: {apriori_time:.2f} seconds")

display(df_itemsets.head())

Frequent itemsets: 191
Apriori execution time: 5.58 seconds


,support,itemsets
0,0.058224,frozenset({Cidre de glace})
1,0.083966,frozenset({Couches})
2,0.143457,frozenset({Fromage cheddar})
3,0.070159,frozenset({Lingettes pour bébé})
4,0.055308,frozenset({Purée pour bébé})


In [19]:
# Generate association rules from my frequent itemsets
MIN_CONFIDENCE = 0.30

df_rules = association_rules(
    df_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

print(f"Frequent itemsets: {len(df_itemsets):,}")
print(f"Association rules: {len(df_rules):,}")

display(
    df_rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

Frequent itemsets: 191
Association rules: 282


,antecedents,consequents,support,confidence,lift
0,frozenset({Cidre de glace}),frozenset({Sac réutilisable}),0.052520,0.902034,1.002231
1,frozenset({Lingettes pour bébé}),frozenset({Couches}),0.055900,0.796762,9.489098
2,frozenset({Couches}),frozenset({Lingettes pour bébé}),0.055900,0.665746,9.489098
3,frozenset({Couches}),frozenset({Sac réutilisable}),0.075561,0.899900,0.999860
4,frozenset({Fromage cheddar}),frozenset({Sac réutilisable}),0.129200,0.900618,1.000658
5,frozenset({Lingettes pour bébé}),frozenset({Sac réutilisable}),0.063087,0.899200,0.999083
6,frozenset({Thon en conserve}),frozenset({Sac réutilisable}),0.086350,0.900971,1.001050
7,frozenset({Thé vert}),frozenset({Sac réutilisable}),0.083499,0.898496,0.998300
8,frozenset({Biscuits}),frozenset({Sac réutilisable}),0.145328,0.900204,1.000198
9,frozenset({Bière}),frozenset({Sac réutilisable}),0.135463,0.900278,1.000280


In [20]:
# Measure Apriori execution time and peak memory
from memory_profiler import memory_usage

start_time = time.perf_counter()

peak_memory, df_itemsets = memory_usage(
    (
        apriori,
        (df_encoded,),
        {
            "min_support": 0.05,
            "use_colnames": True,
            "low_memory": True
        }
    ),
    interval=0.1,
    max_usage=True,
    retval=True
)

apriori_time = time.perf_counter() - start_time

print(f"Frequent itemsets: {len(df_itemsets):,}")
print(f"Apriori execution time: {apriori_time:.2f} seconds")
print(f"Peak process memory: {peak_memory:.2f} MiB")

Frequent itemsets: 191
Apriori execution time: 5.78 seconds
Peak process memory: 1041.11 MiB


In [21]:
# Experiment 2 - Apriori with 2% minimum support
start_time = time.perf_counter()

peak_memory_02, df_itemsets_02 = memory_usage(
    (
        apriori,
        (df_encoded,),
        {
            "min_support": 0.02,
            "use_colnames": True,
            "low_memory": True
        }
    ),
    interval=0.1,
    max_usage=True,
    retval=True
)

apriori_time_02 = time.perf_counter() - start_time

# Generate association rules
df_rules_02 = association_rules(
    df_itemsets_02,
    metric="confidence",
    min_threshold=0.30
)

print(f"Frequent itemsets: {len(df_itemsets_02):,}")
print(f"Association rules: {len(df_rules_02):,}")
print(f"Execution time: {apriori_time_02:.2f} seconds")
print(f"Peak process memory: {peak_memory_02:.2f} MiB")

Frequent itemsets: 658
Association rules: 2,707
Execution time: 11.22 seconds
Peak process memory: 1060.12 MiB


In [22]:
# Experiment 3 - Apriori with 1% minimum support
start_time = time.perf_counter()

peak_memory_01, df_itemsets_01 = memory_usage(
    (
        apriori,
        (df_encoded,),
        {
            "min_support": 0.01,
            "use_colnames": True,
            "low_memory": True
        }
    ),
    interval=0.1,
    max_usage=True,
    retval=True
)

apriori_time_01 = time.perf_counter() - start_time

# Generate association rules
df_rules_01 = association_rules(
    df_itemsets_01,
    metric="confidence",
    min_threshold=0.30
)

print(f"Frequent itemsets: {len(df_itemsets_01):,}")
print(f"Association rules: {len(df_rules_01):,}")
print(f"Execution time: {apriori_time_01:.2f} seconds")
print(f"Peak process memory: {peak_memory_01:.2f} MiB")

Frequent itemsets: 2,267
Association rules: 7,817
Execution time: 24.28 seconds
Peak process memory: 1065.75 MiB


In [23]:
# Experiment 4 - Apriori with 0.5% minimum support
start_time = time.perf_counter()

peak_memory_005, df_itemsets_005 = memory_usage(
    (
        apriori,
        (df_encoded,),
        {
            "min_support": 0.005,
            "use_colnames": True,
            "low_memory": True
        }
    ),
    interval=0.1,
    max_usage=True,
    retval=True
)

apriori_time_005 = time.perf_counter() - start_time

# Generate association rules
df_rules_005 = association_rules(
    df_itemsets_005,
    metric="confidence",
    min_threshold=0.30
)

print(f"Frequent itemsets: {len(df_itemsets_005):,}")
print(f"Association rules: {len(df_rules_005):,}")
print(f"Execution time: {apriori_time_005:.2f} seconds")
print(f"Peak process memory: {peak_memory_005:.2f} MiB")

Frequent itemsets: 7,106
Association rules: 23,918
Execution time: 42.56 seconds
Peak process memory: 1073.48 MiB


In [24]:
# Compare Apriori memory settings on my verified sample
df_test_baskets = df_encoded_sample

for low_memory_setting in [False, True]:
    start_time = time.perf_counter()

    peak_memory, df_test_itemsets = memory_usage(
        (
            apriori,
            (df_test_baskets,),
            {
                "min_support": 0.05,
                "use_colnames": True,
                "low_memory": low_memory_setting
            }
        ),
        interval=0.1,
        max_usage=True,
        retval=True
    )

    elapsed_time = time.perf_counter() - start_time

    print(f"\nlow_memory={low_memory_setting}")
    print(f"Frequent itemsets: {len(df_test_itemsets):,}")
    print(f"Execution time: {elapsed_time:.2f} seconds")
    print(f"Peak process memory: {peak_memory:.2f} MiB")


low_memory=False
Frequent itemsets: 193
Execution time: 2.43 seconds
Peak process memory: 940.74 MiB

low_memory=True
Frequent itemsets: 193
Execution time: 2.48 seconds
Peak process memory: 933.43 MiB


### Phase 3 - Evaluate the association rules

In [25]:
# Keep rules with one antecedent and one consequent
df_simple_rules = df_rules_005[
    (df_rules_005["antecedents"].apply(len) == 1) &
    (df_rules_005["consequents"].apply(len) == 1)
].copy()

print(f"Total simple rules: {len(df_simple_rules):,}")

# Top 10 rules by confidence
print("Top 10 Rules by Confidence")
display(
    df_simple_rules.sort_values(
        "confidence", ascending=False
    )[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

# Top 10 rules by lift
print("Top 10 Rules by Lift")
display(
    df_simple_rules.sort_values(
        "lift", ascending=False
    )[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

Total simple rules: 354
Top 10 Rules by Confidence


,antecedents,consequents,support,confidence,lift
80,frozenset({Concombre}),frozenset({Sac réutilisable}),0.062853,0.902619,1.002881
0,frozenset({Cidre de glace}),frozenset({Sac réutilisable}),0.052520,0.902034,1.002231
37,frozenset({Sauce tomate}),frozenset({Sac réutilisable}),0.095588,0.901808,1.001980
88,frozenset({Relish}),frozenset({Sac réutilisable}),0.053301,0.901695,1.001854
98,frozenset({Huile d'olive}),frozenset({Sac réutilisable}),0.049201,0.901662,1.001818
63,frozenset({Pommes}),frozenset({Sac réutilisable}),0.082351,0.901617,1.001768
73,frozenset({Moutarde de Dijon}),frozenset({Sac réutilisable}),0.074374,0.901525,1.001665
50,frozenset({Fromage suisse}),frozenset({Sac réutilisable}),0.083997,0.901458,1.001592
72,frozenset({Steak de boeuf}),frozenset({Sac réutilisable}),0.095092,0.901414,1.001543
105,frozenset({Avocat}),frozenset({Sac réutilisable}),0.039907,0.901344,1.001464


Top 10 Rules by Lift


,antecedents,consequents,support,confidence,lift
197,frozenset({Fromage à raclette}),frozenset({Bouillon à fondue}),0.033651,0.491198,10.170786
196,frozenset({Bouillon à fondue}),frozenset({Fromage à raclette}),0.033651,0.696780,10.170786
221,frozenset({Charcuteries}),frozenset({Fromage à raclette}),0.031932,0.683403,9.975519
220,frozenset({Fromage à raclette}),frozenset({Charcuteries}),0.031932,0.466106,9.975519
205,frozenset({Fromage à raclette}),frozenset({Cornichons}),0.029733,0.434008,9.772967
206,frozenset({Cornichons}),frozenset({Fromage à raclette}),0.029733,0.669526,9.772967
215,frozenset({Pain baguette rustique}),frozenset({Fromage à raclette}),0.027627,0.654374,9.551784
214,frozenset({Fromage à raclette}),frozenset({Pain baguette rustique}),0.027627,0.403267,9.551784
184,frozenset({Algues nori}),frozenset({Riz à sushi}),0.054431,0.795019,9.504685
185,frozenset({Riz à sushi}),frozenset({Algues nori}),0.054431,0.650738,9.504685



### Rule Evaluation and Interpretation

**Understanding the metrics:**

- **Support:** How often a product or combination of products appears across all baskets.
- **Confidence:** How often customers purchase the recommended product when they have already purchased the first product.
- **Lift:** How much more likely customers are to purchase the products together compared with the normal purchasing frequency.

**Comparing Confidence and Lift:**

The Reusable Bag dominated my confidence rankings because it appears in approximately 90% of all baskets. Even though the confidence was high, its lift was close to 1, meaning the association was not particularly useful.

The lift rankings showed much stronger relationships, such as Raclette Cheese with Fondue Broth, Charcuteries, and Pickles. These combinations had lift values around 9–10, showing a much stronger association compared with their normal purchasing frequency.

For my recommendation engine, I would prioritize lift over confidence alone. However, I would still consider support and confidence to make sure the recommendations are reliable and make sense for the business.

**Pre-Mining Predictions:**

Both predictions were supported by my results:

1. Rare products such as Foie gras, Caviar, Truffle Oil, and Saffron did not appear in the frequent itemsets because their support was below my minimum threshold of 0.5%.
2. Reusable Bag dominated the confidence rankings but had lift values close to 1, confirming that high confidence does not always mean a strong association.

**Filtering the Rules:**

We started with 23,918 association rules and kept 22,476 after filtering for `lift > 1`, removing 1,442 rules.

This helps eliminate rules that do not show a positive association. However, a lift just above 1 may still provide little business value, so I should consider more than one metric when selecting my final recommendations.


In [26]:
# Keep association rules with positive lift
df_filtered_rules = df_rules_005[
    df_rules_005["lift"] > 1
].copy()

print(f"Original rules: {len(df_rules_005):,}")
print(f"Filtered rules: {len(df_filtered_rules):,}")
print(f"Rules removed: {len(df_rules_005) - len(df_filtered_rules):,}")

Original rules: 23,918
Filtered rules: 22,476
Rules removed: 1,442


In [27]:
# Check whether rare products appear in frequent itemsets
rare_products = ["Foie gras", "Caviar", "Huile de truffe", "Safran"]

mined_products = set()

for itemset in df_itemsets_005["itemsets"]:
    mined_products.update(itemset)

for product in rare_products:
    print(f"{product}: {product in mined_products}")

Foie gras: False
Caviar: False
Huile de truffe: False
Safran: False


### Phase 4 - Seasonal analysis

In [28]:
from mlxtend.frequent_patterns import apriori, association_rules

# Identify each basket's date
df_basket_dates = df_transactions[
    ["TransactionID", "Date"]
].drop_duplicates()

# Each basket should have exactly one date
assert df_basket_dates["TransactionID"].is_unique

df_basket_dates["Date"] = pd.to_datetime(
    df_basket_dates["Date"]
)

# Align basket dates with the encoded transaction matrix
basket_dates = df_basket_dates.set_index("TransactionID")[
    "Date"
].reindex(df_encoded.index)

assert basket_dates.notna().all()

print("Date range:", basket_dates.min(), "to", basket_dates.max())


# Separate summer and winter baskets
# Summer: June, July, August
df_summer_encoded = df_encoded.loc[
    basket_dates.dt.month.isin([6, 7, 8])
]

# Winter: December, January, February
df_winter_encoded = df_encoded.loc[
    basket_dates.dt.month.isin([12, 1, 2])
]

assert len(df_summer_encoded) > 0
assert len(df_winter_encoded) > 0

print(f"Summer baskets: {len(df_summer_encoded):,}")
print(f"Winter baskets: {len(df_winter_encoded):,}")


# Mine frequent itemsets separately
MIN_SUPPORT = 0.005
MIN_CONFIDENCE = 0.30

df_summer_itemsets = apriori(
    df_summer_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    low_memory=True
)

df_winter_itemsets = apriori(
    df_winter_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    low_memory=True
)

print(f"Summer frequent itemsets: {len(df_summer_itemsets):,}")
print(f"Winter frequent itemsets: {len(df_winter_itemsets):,}")


# Generate and filter seasonal rules
df_summer_rules = association_rules(
    df_summer_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

df_winter_rules = association_rules(
    df_winter_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

# Apply the same lift filter as the year-round model
df_summer_rules = df_summer_rules[
    df_summer_rules["lift"] > 1
].copy()

df_winter_rules = df_winter_rules[
    df_winter_rules["lift"] > 1
].copy()

print(f"Summer filtered rules: {len(df_summer_rules):,}")
print(f"Winter filtered rules: {len(df_winter_rules):,}")


# Compare identical rules across both seasons
metrics = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]

df_seasonal_comparison = df_summer_rules[metrics].merge(
    df_winter_rules[metrics],
    on=["antecedents", "consequents"],
    how="outer",
    suffixes=("_summer", "_winter"),
    indicator=True
)

# Rules that exist in both seasonal rule sets
df_shared_rules = df_seasonal_comparison[
    df_seasonal_comparison["_merge"] == "both"
].copy()

# Rules retained in only one season
df_summer_only = df_seasonal_comparison[
    df_seasonal_comparison["_merge"] == "left_only"
].copy()

df_winter_only = df_seasonal_comparison[
    df_seasonal_comparison["_merge"] == "right_only"
].copy()

print(f"Rules shared by both seasons: {len(df_shared_rules):,}")
print(f"Summer-only rules: {len(df_summer_only):,}")
print(f"Winter-only rules: {len(df_winter_only):,}")


# Compare support, confidence and lift
df_shared_rules["lift_difference"] = (
    df_shared_rules["lift_summer"]
    - df_shared_rules["lift_winter"]
).abs()

display(
    df_shared_rules.sort_values(
        "lift_difference",
        ascending=False
    )[
        [
            "antecedents",
            "consequents",
            "support_summer",
            "support_winter",
            "confidence_summer",
            "confidence_winter",
            "lift_summer",
            "lift_winter"
        ]
    ].head(10)
)


# Find an example of a season-specific rule
if not df_summer_only.empty:

    example_rule = df_summer_only.sort_values(
        "lift_summer",
        ascending=False
    ).iloc[0]

    source_season = "Summer"
    other_season = "Winter"
    df_other_season = df_winter_encoded

else:

    example_rule = df_winter_only.sort_values(
        "lift_winter",
        ascending=False
    ).iloc[0]

    source_season = "Winter"
    other_season = "Summer"
    df_other_season = df_summer_encoded


print(f"Example rule found in {source_season} but not {other_season}:")
print("Antecedents:", example_rule["antecedents"])
print("Consequents:", example_rule["consequents"])

# Measure the same rule in the opposite season
antecedents = example_rule["antecedents"]
consequents = example_rule["consequents"]

support_antecedents = df_other_season[
    list(antecedents)
].all(axis=1).mean()

support_consequents = df_other_season[
    list(consequents)
].all(axis=1).mean()

support_both = df_other_season[
    list(antecedents | consequents)
].all(axis=1).mean()

confidence_other = (
    support_both / support_antecedents
    if support_antecedents > 0
    else float("nan")
)

lift_other = (
    confidence_other / support_consequents
    if support_consequents > 0
    else float("nan")
)

print(f"\nSame rule measured in {other_season}:")
print(f"Support: {support_both:.6f}")
print(f"Confidence: {confidence_other:.6f}")
print(f"Lift: {lift_other:.6f}")

Date range: 2026-01-01 00:00:00 to 2026-12-31 00:00:00
Summer baskets: 252,474
Winter baskets: 246,965
Summer frequent itemsets: 9,867
Winter frequent itemsets: 6,042
Summer filtered rules: 52,117
Winter filtered rules: 13,876
Rules shared by both seasons: 8,923
Summer-only rules: 43,194
Winter-only rules: 4,953


,antecedents,consequents,support_summer,support_winter,confidence_summer,confidence_winter,lift_summer,lift_winter
50397,"frozenset({Pain à hamburger, Ketchup, Steak de...","frozenset({Charbon de bois, Sauce BBQ})",0.037671,0.005199,0.485428,0.513600,3.844838,29.979018
32500,"frozenset({Ketchup, Pain à hamburger})","frozenset({Charbon de bois, Steak de boeuf, Sa...",0.037671,0.005199,0.349348,0.365292,3.838845,29.921805
56905,"frozenset({Charbon de bois, Steak de boeuf, Sa...","frozenset({Ketchup, Pain à hamburger})",0.037671,0.005199,0.413954,0.425871,3.838845,29.921805
51330,"frozenset({Charbon de bois, Ketchup, Pain à ha...","frozenset({Steak de boeuf, Sauce BBQ})",0.037671,0.005199,0.350184,0.373364,3.837343,29.763695
35718,"frozenset({Steak de boeuf, Sauce BBQ})","frozenset({Charbon de bois, Ketchup, Pain à ha...",0.037671,0.005199,0.412804,0.414461,3.837343,29.763695
54009,"frozenset({Pain à hamburger, Steak de boeuf, S...","frozenset({Charbon de bois, Ketchup})",0.037671,0.005199,0.604103,0.621792,3.850163,29.650667
50747,"frozenset({Charbon de bois, Pain à hamburger, ...","frozenset({Ketchup, Steak de boeuf})",0.037671,0.005199,0.435625,0.452431,3.831260,29.614284
37381,"frozenset({Ketchup, Steak de boeuf})","frozenset({Charbon de bois, Pain à hamburger, ...",0.037671,0.005199,0.331313,0.340313,3.831260,29.614284
57065,"frozenset({Ketchup, Steak de boeuf, Sauce BBQ})","frozenset({Charbon de bois, Pain à hamburger})",0.037671,0.005199,0.684146,0.689581,3.861163,29.281705
52726,"frozenset({Charbon de bois, Ketchup, Steak de ...","frozenset({Pain à hamburger, Sauce BBQ})",0.037671,0.005199,0.332146,0.346465,3.830892,29.143316


Example rule found in Summer but not Winter:
Antecedents: frozenset({'Farine tout usage', 'Oeufs', 'Sac réutilisable'})
Consequents: frozenset({'Bananes', "Sirop d'érable"})

Same rule measured in Winter:
Support: 0.004717
Confidence: 0.298106
Lift: 16.671617


In [29]:
# Compare the selected season-specific rule

print("Selected summer-only rule:")
print("Antecedents:", sorted(example_rule["antecedents"]))
print("Consequents:", sorted(example_rule["consequents"]))

print(f"\nSummer support: {example_rule['support_summer']:.4%}")
print(f"Summer confidence: {example_rule['confidence_summer']:.4%}")
print(f"Summer lift: {example_rule['lift_summer']:.4f}")

print(f"\nWinter support: {support_both:.4%}")
print(f"Winter confidence: {confidence_other:.4%}")
print(f"Winter lift: {lift_other:.4f}")

assert support_both < MIN_SUPPORT
assert confidence_other < MIN_CONFIDENCE

print("\nSeasonal threshold comparison: PASSED")

Selected summer-only rule:
Antecedents: ['Farine tout usage', 'Oeufs', 'Sac réutilisable']
Consequents: ['Bananes', "Sirop d'érable"]

Summer support: 0.5054%
Summer confidence: 30.9558%
Summer lift: 16.8475

Winter support: 0.4717%
Winter confidence: 29.8106%
Winter lift: 16.6716

Seasonal threshold comparison: PASSED


### How often should Boréal Marché re-mine its association rules?

I would recommend re-mining the association rules every three months as a starting point. The seasonal analysis showed a noticeable difference between summer and winter, with 52,117 filtered rules in summer compared to 13,876 in winter.

This suggests that customer purchasing patterns can change depending on the season. I would also monitor support, confidence, and lift monthly to identify changes between mining runs, rather than waiting until the next scheduled update.

### What happens if the website serves year-round rules in January?

If Boréal Marché continues using year-round rules in January, some recommendations might not reflect what customers are currently buying.

For example, the analysis identified rules that met the minimum support and confidence thresholds in summer but fell below those thresholds in winter. This means the website could recommend combinations that were more relevant during another season, while missing opportunities to recommend products that are more relevant in winter.

### Conclusion

Based on these results, I think seasonal rule updates would help keep the recommendations more relevant to customers. However, maintaining seasonal models would also require additional processing, testing, and maintenance.

Quarterly updates seem like a reasonable starting point, but I would continue monitoring the results to determine whether more frequent updates are actually needed.

### Phase 5 - Save filtered association rules

In [30]:
import json

MODEL_DIR = Path("..") / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Prepare the rules for JSON
df_export_rules = df_filtered_rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].copy()

# Convert frozensets into JSON-compatible lists
df_export_rules["antecedents"] = df_export_rules["antecedents"].apply(sorted)
df_export_rules["consequents"] = df_export_rules["consequents"].apply(sorted)

# Save the rules
with open(MODEL_DIR / "rules.json", "w", encoding="utf-8") as f:
    json.dump(
        df_export_rules.to_dict(orient="records"),
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Rules exported: {len(df_export_rules):,}")
print(f"File saved: {MODEL_DIR / 'rules.json'}")

Rules exported: 22,476
File saved: ..\models\rules.json


In [31]:
# Reload the saved association rules
with open(MODEL_DIR / "rules.json", "r", encoding="utf-8") as f:
    loaded_rules = json.load(f)

# Convert JSON records back into a DataFrame
df_loaded_rules = pd.DataFrame(loaded_rules)

# Restore lists back into frozensets
df_loaded_rules["antecedents"] = df_loaded_rules["antecedents"].apply(frozenset)
df_loaded_rules["consequents"] = df_loaded_rules["consequents"].apply(frozenset)

# Verify the saved data matches the original rules
columns_to_check = [
    "antecedents", "consequents", "support", "confidence", "lift"
]

assert len(df_loaded_rules) == len(df_filtered_rules)

assert df_loaded_rules[columns_to_check].equals(
    df_filtered_rules[columns_to_check].reset_index(drop=True)
)

print(f"Rules reloaded: {len(df_loaded_rules):,}")
print("JSON verification: PASSED")

display(df_loaded_rules.head())

Rules reloaded: 22,476
JSON verification: PASSED


,antecedents,consequents,support,confidence,lift
0,frozenset({Cidre de glace}),frozenset({Sac réutilisable}),0.052520,0.902034,1.002231
1,frozenset({Lingettes pour bébé}),frozenset({Couches}),0.055900,0.796762,9.489098
2,frozenset({Couches}),frozenset({Lingettes pour bébé}),0.055900,0.665746,9.489098
3,frozenset({Purée pour bébé}),frozenset({Couches}),0.040908,0.739640,8.808802
4,frozenset({Couches}),frozenset({Purée pour bébé}),0.040908,0.487197,8.808802


In [32]:
# Calculate recommendation coverage

# Keep rules that recommend one product
df_recommendation_rules = df_loaded_rules[
    df_loaded_rules["consequents"].apply(len) == 1
].copy()

# Collect unique products that can be recommended
recommendable_products = set()

for itemset in df_recommendation_rules["consequents"]:
    recommendable_products.update(itemset)

# Compare against the complete catalogue
catalogue_products = set(df_products["Item"])

covered_products = recommendable_products.intersection(catalogue_products)

coverage = len(covered_products) / len(catalogue_products) * 100

print(f"Saved rules: {len(df_loaded_rules):,}")
print(f"Single-consequent rules: {len(df_recommendation_rules):,}")
print(f"Recommendable products: {len(covered_products)} / {len(catalogue_products)}")
print(f"Recommendation coverage: {coverage:.2f}%")

Saved rules: 22,476
Single-consequent rules: 10,909
Recommendable products: 58 / 98
Recommendation coverage: 59.18%


In [33]:
# Save recommendation system metadata
from datetime import date

metadata = {
    "min_support": 0.005,
    "min_confidence": 0.30,
    "lift_filter": "lift > 1",
    "mining_date": date.today().isoformat(),
    "number_of_rules": len(df_loaded_rules),
    "single_consequent_rules": len(df_recommendation_rules),
    "catalogue_products": len(catalogue_products),
    "recommendable_products": len(covered_products),
    "coverage_percent": round(coverage, 2)
}

# Save metadata to JSON
with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# Reload and verify
with open(MODEL_DIR / "metadata.json", "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

assert loaded_metadata == metadata

print("Metadata saved and verified!")
print(f"Rules: {metadata['number_of_rules']:,}")
print(f"Coverage: {metadata['coverage_percent']:.2f}%")

display(loaded_metadata)

Metadata saved and verified!
Rules: 22,476
Coverage: 59.18%


{'min_support': 0.005,
 'min_confidence': 0.3,
 'lift_filter': 'lift > 1',
 'mining_date': '2026-11-04',
 'number_of_rules': 22476,
 'single_consequent_rules': 10909,
 'catalogue_products': 98,
 'recommendable_products': 58,
 'coverage_percent': 59.18}

In [34]:
test_basket = {"Charbon de bois"}

# Find association rules that match the customer's basket
df_matching_rules = df_loaded_rules[
    df_loaded_rules["antecedents"].apply(
        lambda itemset: itemset.issubset(test_basket)
    )
].copy()

# Keep rules that recommend only one product
df_matching_rules = df_matching_rules[
    df_matching_rules["consequents"].apply(len) == 1
]

# Exclude products already in the basket
df_matching_rules = df_matching_rules[
    df_matching_rules["consequents"].apply(
        lambda itemset: itemset.isdisjoint(test_basket)
    )
]

print(f"Matching rules: {len(df_matching_rules):,}")

display(
    df_matching_rules.sort_values(
        "lift", ascending=False
    )[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

Matching rules: 8


,antecedents,consequents,support,confidence,lift
241,frozenset({Charbon de bois}),frozenset({Steak de boeuf}),0.092118,0.655360,6.212410
100,frozenset({Charbon de bois}),frozenset({Pain à hamburger}),0.087373,0.621602,6.157097
240,frozenset({Charbon de bois}),frozenset({Ketchup}),0.077344,0.550252,6.058045
237,frozenset({Charbon de bois}),frozenset({Saucisses}),0.070764,0.503440,5.983287
211,frozenset({Charbon de bois}),frozenset({Sauce BBQ}),0.062042,0.441388,5.844037
243,frozenset({Charbon de bois}),frozenset({Relish}),0.045494,0.323660,5.475372
69,frozenset({Charbon de bois}),frozenset({Bière}),0.068263,0.485647,3.227575
42,frozenset({Charbon de bois}),frozenset({Sac réutilisable}),0.126552,0.900335,1.000343


In [35]:
# Adapt my homework recommendation function
def recommend(basket, df_rules, top_n=5):

    basket = set(basket)

    # Find rules that match the basket
    df_matching = df_rules[
        df_rules["antecedents"].apply(
            lambda itemset: itemset.issubset(basket)
        )
    ].copy()

    if df_matching.empty:
        return df_matching

    # Keep rules that recommend one product
    df_matching = df_matching[
        df_matching["consequents"].apply(len) == 1
    ].copy()

    if df_matching.empty:
        return df_matching

    # Exclude products already in the basket
    df_matching = df_matching[
        df_matching["consequents"].apply(
            lambda itemset: itemset.isdisjoint(basket)
        )
    ].copy()

    if df_matching.empty:
        return df_matching

    # Identify each recommended product
    df_matching["product"] = df_matching["consequents"].apply(
        lambda itemset: next(iter(itemset))
    )

    # Prioritize lift, then confidence and support
    df_matching = df_matching.sort_values(
        ["lift", "confidence", "support"],
        ascending=False
    )

    # Keep the strongest rule for each product
    df_matching = df_matching.drop_duplicates(
        subset="product",
        keep="first"
    )

    return df_matching.head(top_n)

In [36]:
# Test the recommendation engine
df_recommendations = recommend(
    {"Charbon de bois"},
    df_loaded_rules,
    top_n=5
)

display(
    df_recommendations[
        ["product", "antecedents", "support", "confidence", "lift"]
    ]
)

# QA assertions
assert len(df_recommendations) <= 5
assert df_recommendations["product"].is_unique
assert "Charbon de bois" not in df_recommendations["product"].values
assert df_recommendations["lift"].is_monotonic_decreasing

print("Recommendation engine QA: PASSED")

,product,antecedents,support,confidence,lift
241,Steak de boeuf,frozenset({Charbon de bois}),0.092118,0.655360,6.212410
100,Pain à hamburger,frozenset({Charbon de bois}),0.087373,0.621602,6.157097
240,Ketchup,frozenset({Charbon de bois}),0.077344,0.550252,6.058045
237,Saucisses,frozenset({Charbon de bois}),0.070764,0.503440,5.983287
211,Sauce BBQ,frozenset({Charbon de bois}),0.062042,0.441388,5.844037


Recommendation engine QA: PASSED


In [37]:
# Test recommendations with multiple products
test_basket_multi = {
    "Charbon de bois",
    "Pain à hamburger",
    "Steak de boeuf"
}

df_multi_recommendations = recommend(
    test_basket_multi,
    df_loaded_rules,
    top_n=5
)

display(
    df_multi_recommendations[
        ["product", "antecedents", "support", "confidence", "lift"]
    ]
)

# QA checks
assert len(df_multi_recommendations) <= 5
assert df_multi_recommendations["product"].is_unique
assert not set(df_multi_recommendations["product"]).intersection(test_basket_multi)
assert df_multi_recommendations["lift"].is_monotonic_decreasing

print("Multi-item basket QA: PASSED")

,product,antecedents,support,confidence,lift
10946,Ketchup,"frozenset({Charbon de bois, Pain à hamburger, ...",0.038293,0.607392,6.687125
10895,Saucisses,"frozenset({Charbon de bois, Pain à hamburger, ...",0.035161,0.557713,6.628311
10734,Sauce BBQ,"frozenset({Charbon de bois, Pain à hamburger, ...",0.030697,0.486906,6.446698
11010,Relish,"frozenset({Charbon de bois, Pain à hamburger, ...",0.022469,0.356396,6.029169
2364,Bière,"frozenset({Charbon de bois, Steak de boeuf})",0.048445,0.525902,3.495106


Multi-item basket QA: PASSED


In [38]:
# Identify matching rules with multiple antecedents
df_multi_rules = df_loaded_rules[
    (df_loaded_rules["antecedents"].apply(len) > 1) &
    (df_loaded_rules["antecedents"].apply(
        lambda itemset: itemset.issubset(test_basket_multi)
    ))
]

print(f"Multi-item rules triggered: {len(df_multi_rules):,}")

display(
    df_multi_rules[
        ["antecedents", "consequents", "confidence", "lift"]
    ].head()
)

Multi-item rules triggered: 89


,antecedents,consequents,confidence,lift
897,"frozenset({Pain à hamburger, Steak de boeuf})",frozenset({Sac réutilisable}),0.900267,1.000267
1734,"frozenset({Charbon de bois, Steak de boeuf})",frozenset({Sac réutilisable}),0.901051,1.001139
2239,"frozenset({Charbon de bois, Pain à hamburger})",frozenset({Bière}),0.524029,3.482662
2253,"frozenset({Pain à hamburger, Steak de boeuf})",frozenset({Bière}),0.523126,3.476662
2364,"frozenset({Charbon de bois, Steak de boeuf})",frozenset({Bière}),0.525902,3.495106


In [39]:
# Verify that the strongest rule is kept for each recommendation

df_candidates = df_loaded_rules[
    df_loaded_rules["antecedents"].apply(
        lambda itemset: itemset.issubset(test_basket_multi)
    ) &
    (df_loaded_rules["consequents"].apply(len) == 1) &
    df_loaded_rules["consequents"].apply(
        lambda itemset: itemset.isdisjoint(test_basket_multi)
    )
].copy()

df_candidates["product"] = df_candidates["consequents"].apply(
    lambda itemset: next(iter(itemset))
)

# Find the maximum lift for each recommended product
best_lift = df_candidates.groupby("product")["lift"].max()

# Verify that duplicate recommendation candidates exist
assert df_candidates["product"].duplicated().any()

# Verify that each selected recommendation has the highest lift
for _, row in df_multi_recommendations.iterrows():
    assert row["lift"] == best_lift.loc[row["product"]]

print("Highest-lift rule selection: PASSED")


Highest-lift rule selection: PASSED


In [40]:
print(sorted(df_products["Category"].dropna().unique()))

['Boissons', 'Boulangerie', 'Bébé', 'Collations', 'Fromages', 'Fruits et légumes', 'Maison', 'Poissons', 'Produits laitiers', 'Spécialités', 'Surgelés', 'Viandes', 'Épicerie']


In [41]:
# Export product popularity for fallback recommendations

df_popularity = pd.DataFrame({
    "Item": df_encoded.columns,
    "basket_count": df_encoded.sum(axis=0).to_numpy()
})

# Percentage of baskets containing each product
df_popularity["support"] = (
    df_popularity["basket_count"] / len(df_encoded)
)

# Most popular products first
df_popularity = df_popularity.sort_values(
    ["basket_count", "Item"],
    ascending=[False, True]
).reset_index(drop=True)

# Save popularity data for the application
popularity_path = MODEL_DIR / "product_popularity.json"

with open(popularity_path, "w", encoding="utf-8") as file:
    json.dump(
        df_popularity.to_dict(orient="records"),
        file,
        ensure_ascii=False,
        indent=2
    )

# QA verification
assert len(df_popularity) == len(df_products)
assert df_popularity["Item"].is_unique
assert set(df_popularity["Item"]) == set(df_products["Item"])
assert int(df_popularity["basket_count"].sum()) == len(df_transactions)

with open(popularity_path, "r", encoding="utf-8") as file:
    loaded_popularity = json.load(file)

assert len(loaded_popularity) == len(df_products)

print("Product popularity export: PASSED")

display(df_popularity.head(10))
display(df_popularity.tail(5))


Product popularity export: PASSED


,Item,basket_count,support
0,Sac réutilisable,900026,0.900026
1,Biscuits,161439,0.161439
2,Noix mélangées,155189,0.155189
3,Beurre d'arachide,153726,0.153726
4,Bière,150468,0.150468
5,Savon à vaisselle,149820,0.149820
6,Fromage cheddar,143457,0.143457
7,Charbon de bois,140561,0.140561
8,Papier hygiénique,139758,0.139758
9,Céréales,139198,0.139198


,Item,basket_count,support
93,Safran,818,0.000818
94,Huile de truffe,817,0.000817
95,Caviar,810,0.000810
96,Vinaigre balsamique vieilli,779,0.000779
97,Foie gras,756,0.000756
